# Benchmarks - Python

The one Python example from [docs/benchmarks.md](https://platob.github.io/yggdryl/benchmarks/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

### The pipeline those numbers measure

In [ ]:
import gzip
import pathlib
import shutil
import tempfile

import pyarrow as pa
import pyarrow.compute as pc

from yggdryl import IOBase, combined, schema_from_pattern, zstd
from yggdryl.iceberg import Catalog

pattern = (
    r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\S*"
    r" \[(?<level>[^\]]+)\] \[(?<logger>[^\]]+)\]"
    r" \[(?<thread_id>\d+)\] took=(?<latency_us>\d+)"
)
# The older extractor: same records, no thread column.
archived_pattern = pattern.replace(r"\[(?<thread_id>\d+)\]", r"\[\d+\]")

extractor = {
    "pattern": pattern,
    "byte_size": 8 * 1024 * 1024,
    "custom_fields": {"source": "gateway"},
}
older = {"pattern": archived_pattern, "custom_fields": {"source": "archive"}}

root = pathlib.Path(tempfile.mkdtemp(prefix="yggdryl-doc-"))
incoming = root / "incoming"
archive = root / "archive"
incoming.mkdir()
archive.mkdir()

# Three rotated leaves in three codings; the second record spans a stack trace.
(incoming / "app-0.log.gz").write_bytes(
    gzip.compress(
        b"2024-02-01 10:00:00.000000 [ii] [engine] [3] took=120 fill 100 SYMB-0001\n"
        b"2024-02-01 10:00:01.000000 [ee] [engine] [4] took=980 fill 101 SYMB-0002\n"
        b"    at engine::match(order.rs:118)\n"
        b"    at engine::step(order.rs:64)\n"
        b"2024-02-01 10:00:02.000000 [ww] [router] [5] took=240 fill 102 SYMB-0003\n"
    )
)
(incoming / "app-1.log").write_bytes(
    b"2024-02-01 10:00:03.000000 [ee] [ledger] [6] took=770 fill 103 SYMB-0004\n"
)
(incoming / "app-2.log.zst").write_bytes(
    zstd.dumps(b"2024-02-01 10:00:04.000000 [ii] [feed] [7] took=100 fill 104 SYMB-0005\n")
)
(archive / "app-9.log.gz").write_bytes(
    gzip.compress(
        b"2024-01-31 23:59:59.000000 [ee] [engine] [2] took=310 fill 099 SYMB-0000\n"
    )
)

# 1. The table exists before the first record does.
marked = schema_from_pattern(options=extractor).with_partition_fields(["level"])
catalog = Catalog(root / "warehouse")
table = catalog.tables.create("logs.app", marked)

# 2, 3, 4. One handle per folder, and one lazy combine over the two: both
# schemas are answered without pulling a batch, so nothing is read yet.
stream = combined(
    IOBase(incoming).read_arrow_lines(options=extractor),
    IOBase(archive).read_arrow_lines(options=older),
)

# 5. One commit, handed the reader itself - never a list of batches.
table.append(stream)

# The read-back asserts on the table, not on anything held in memory.
rows = table.scan().read_all()
# Five live records - not the seven lines they occupy - and one archived.
assert rows.num_rows == 6
assert pc.sum(rows.column("latency_us")).as_py() == 120 + 980 + 240 + 770 + 100 + 310
assert rows.schema.field("latency_us").type == pa.int64()
assert rows.column("thread_id").null_count == 1

reopened = catalog.table("logs.app")
assert [field.name for field in reopened.spec.fields] == ["level"]
assert set(rows.column("source").to_pylist()) == {"gateway", "archive"}

shutil.rmtree(root)